In [1]:
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

print(" Imports OK")

 Imports OK


In [ ]:
# optimisation de la RAM pour éviter les erreurs de mémoire
def reduce_mem_usage(df, verbose=True):
    """Réduit l'usage mémoire en convertissant les types de données"""
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32)
    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Mémoire réduite de {start_mem:.2f} MB à {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    
    return df

In [3]:
# chargement minimal et optimisé
data_dir = 'data/'

# Application (seulement colonnes essentielles)
cols_app = [
    'SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

print("Chargement application_train...")
app_train = pd.read_csv(data_dir + 'application_train.csv', usecols=cols_app)
app_train = reduce_mem_usage(app_train)

print("Chargement application_test...")
cols_app_test = [c for c in cols_app if c != 'TARGET']
app_test = pd.read_csv(data_dir + 'application_test.csv', usecols=cols_app_test)
app_test = reduce_mem_usage(app_test)

print(f" Train: {app_train.shape}, Test: {app_test.shape}")

Chargement application_train...
Mémoire réduite de 42.23 MB à 26.39 MB (37.5% reduction)
Chargement application_test...
Mémoire réduite de 6.32 MB à 4.14 MB (34.6% reduction)
 Train: (307511, 18), Test: (48744, 17)


In [4]:
# feature engineering selon le kernel du kaggle
# Correction anomalies
app_train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
app_test['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# Ratios essentiels
app_train['CREDIT_INCOME_RATIO'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']
app_test['CREDIT_INCOME_RATIO'] = app_test['AMT_CREDIT'] / app_test['AMT_INCOME_TOTAL']

app_train['ANNUITY_INCOME_RATIO'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
app_test['ANNUITY_INCOME_RATIO'] = app_test['AMT_ANNUITY'] / app_test['AMT_INCOME_TOTAL']

app_train['DAYS_EMPLOYED_PERC'] = app_train['DAYS_EMPLOYED'] / app_train['DAYS_BIRTH']
app_test['DAYS_EMPLOYED_PERC'] = app_test['DAYS_EMPLOYED'] / app_test['DAYS_BIRTH']

print("✅ Features créées")

✅ Features créées


In [ ]:
# bureau et bureau_balance
bureau_cols = ['SK_ID_CURR', 'SK_ID_BUREAU', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 
               'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT']
bureau = pd.read_csv(data_dir + 'bureau.csv', usecols=bureau_cols)
bureau = reduce_mem_usage(bureau, verbose=False)

# Convertir toutes les colonnes en numérique (certaines peuvent être en object)
for col in ['DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT']:
    bureau[col] = pd.to_numeric(bureau[col], errors='coerce')

# Agrégation minimale
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'DAYS_CREDIT': ['mean', 'min'],
    'CREDIT_DAY_OVERDUE': ['mean', 'max'],
    'AMT_CREDIT_SUM': ['mean', 'sum'],
    'AMT_CREDIT_SUM_DEBT': ['mean', 'sum'],
    'SK_ID_BUREAU': 'count'
})
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg.reset_index(inplace=True)

print(f" Bureau: {bureau_agg.shape}")
del bureau
gc.collect()

✅ Bureau: (305811, 10)


0

In [6]:
# previous_application
prev_cols = ['SK_ID_CURR', 'SK_ID_PREV', 'AMT_CREDIT', 'AMT_ANNUITY', 
             'DAYS_DECISION', 'CNT_PAYMENT']
previous = pd.read_csv(data_dir + 'previous_application.csv', usecols=prev_cols)
previous = reduce_mem_usage(previous, verbose=False)

# Convertir en numérique
for col in ['AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_DECISION', 'CNT_PAYMENT']:
    previous[col] = pd.to_numeric(previous[col], errors='coerce')

prev_agg = previous.groupby('SK_ID_CURR').agg({
    'AMT_CREDIT': ['mean', 'max'],
    'AMT_ANNUITY': ['mean', 'max'],
    'DAYS_DECISION': ['mean', 'min'],
    'CNT_PAYMENT': ['mean', 'sum'],
    'SK_ID_PREV': 'count'
})
prev_agg.columns = ['PREV_' + '_'.join(col).upper() for col in prev_agg.columns]
prev_agg.reset_index(inplace=True)

print(f" Previous: {prev_agg.shape}")
del previous
gc.collect()

 Previous: (338857, 10)


0

In [7]:
#installments_payments
install_cols = ['SK_ID_CURR', 'SK_ID_PREV', 'AMT_PAYMENT', 'AMT_INSTALMENT',
                'DAYS_ENTRY_PAYMENT', 'DAYS_INSTALMENT']
installments = pd.read_csv(data_dir + 'installments_payments.csv', usecols=install_cols)
installments = reduce_mem_usage(installments, verbose=False)

# Convertir en numérique
for col in ['AMT_PAYMENT', 'AMT_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'DAYS_INSTALMENT']:
    installments[col] = pd.to_numeric(installments[col], errors='coerce')

# Features simples
installments['PAYMENT_DIFF'] = installments['AMT_PAYMENT'] - installments['AMT_INSTALMENT']
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']

install_agg = installments.groupby('SK_ID_CURR').agg({
    'AMT_PAYMENT': ['mean', 'sum'],
    'PAYMENT_DIFF': ['mean', 'min'],
    'DAYS_LATE': ['mean', 'max'],
    'SK_ID_PREV': 'count'
})
install_agg.columns = ['INSTAL_' + '_'.join(col).upper() for col in install_agg.columns]
install_agg.reset_index(inplace=True)

print(f" Installments: {install_agg.shape}")
del installments
gc.collect()

 Installments: (339587, 8)


0

In [8]:
#POS_CASH_balance
pos_cols = ['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'SK_DPD']
pos_cash = pd.read_csv(data_dir + 'POS_CASH_balance.csv', usecols=pos_cols)
pos_cash = reduce_mem_usage(pos_cash, verbose=False)

# Convertir en numérique
for col in ['MONTHS_BALANCE', 'CNT_INSTALMENT', 'SK_DPD']:
    pos_cash[col] = pd.to_numeric(pos_cash[col], errors='coerce')

pos_agg = pos_cash.groupby('SK_ID_CURR').agg({
    'MONTHS_BALANCE': ['min', 'max'],
    'CNT_INSTALMENT': ['mean', 'max'],
    'SK_DPD': ['mean', 'max'],
    'SK_ID_PREV': 'count'
})
pos_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_agg.columns]
pos_agg.reset_index(inplace=True)

print(f" POS_CASH: {pos_agg.shape}")
del pos_cash
gc.collect()

 POS_CASH: (337252, 8)


0

In [9]:
# credit_card_balance
cc_cols = ['SK_ID_CURR', 'SK_ID_PREV', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'SK_DPD']
credit_card = pd.read_csv(data_dir + 'credit_card_balance.csv', usecols=cc_cols)
credit_card = reduce_mem_usage(credit_card, verbose=False)

# Convertir en numérique
for col in ['AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'SK_DPD']:
    credit_card[col] = pd.to_numeric(credit_card[col], errors='coerce')

cc_agg = credit_card.groupby('SK_ID_CURR').agg({
    'AMT_BALANCE': ['mean', 'max'],
    'AMT_CREDIT_LIMIT_ACTUAL': ['mean', 'max'],
    'SK_DPD': ['mean', 'max'],
    'SK_ID_PREV': 'count'
})
cc_agg.columns = ['CC_' + '_'.join(col).upper() for col in cc_agg.columns]
cc_agg.reset_index(inplace=True)

print(f" Credit Card: {cc_agg.shape}")
del credit_card
gc.collect()

 Credit Card: (103558, 8)


0

In [10]:
# fusion avec train set et test_set
train = app_train.copy()
test = app_test.copy()

# Fusions successives
for df_name, df_agg in [('Bureau', bureau_agg), ('Previous', prev_agg), 
                         ('Installments', install_agg), ('POS', pos_agg), ('CC', cc_agg)]:
    train = train.merge(df_agg, on='SK_ID_CURR', how='left')
    test = test.merge(df_agg, on='SK_ID_CURR', how='left')
    print(f"Après {df_name}: {train.shape}")

del bureau_agg, prev_agg, install_agg, pos_agg, cc_agg
gc.collect()

print(f"\n Fusion - Train: {train.shape}, Test: {test.shape}")

Après Bureau: (307511, 30)
Après Previous: (307511, 39)
Après Installments: (307511, 46)
Après POS: (307511, 53)
Après CC: (307511, 60)

 Fusion - Train: (307511, 60), Test: (48744, 59)


In [11]:
# encodage simple
# Label encoding seulement (pas de One-Hot)
le = LabelEncoder()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    train[col] = train[col].fillna('Missing').astype(str)
    test[col] = test[col].fillna('Missing').astype(str)
    combined = pd.concat([train[col], test[col]])
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

print(f" Encodage - Train: {train.shape}, Test: {test.shape}")

 Encodage - Train: (307511, 60), Test: (48744, 59)


In [12]:
# preparation de X et y
# Aligner colonnes
train_labels = train['TARGET']
train = train.drop(columns=['TARGET'])
train, test = train.align(test, join='inner', axis=1)
train['TARGET'] = train_labels

# Préparation X, y
X = train.drop(columns=['TARGET', 'SK_ID_CURR']).fillna(-999)
y = train['TARGET']
X_test = test.drop(columns=['SK_ID_CURR']).fillna(-999)

print(f" X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f" Nombre de features: {X.shape[1]}")

 X: (307511, 58), y: (307511,), X_test: (48744, 58)
 Nombre de features: 58


In [36]:
# Modelisation
# Logistic Regression en premier (baseline simple)
lr = LogisticRegression(max_iter=500, random_state=42, solver='liblinear')

print("Validation croisée...")
cv_scores_lr = cross_val_score(lr, X, y, cv=3, scoring='roc_auc', n_jobs=1)
print(f"ROC AUC Logistic Regression: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")

print("Entraînement final...")
lr.fit(X, y)
predictions_lr = lr.predict_proba(X_test)[:, 1]

print(" Modèle entraîné en Logistic Regression")

Validation croisée...
ROC AUC Logistic Regression: 0.6731 (+/- 0.0020)
Entraînement final...
 Modèle entraîné en Logistic Regression


In [28]:
# Random Forest 
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    random_state=42, 
    n_jobs=-1,
    verbose=0  # Pas de logs
)

print("Validation croisée Random Forest...")
cv_scores_rf = cross_val_score(rf, X, y, cv=3, scoring='roc_auc', n_jobs=1)
print(f"ROC AUC RF: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})")

print("Entraînement final...")
rf.fit(X, y)
predictions_rf = rf.predict_proba(X_test)[:, 1]
print(" Random Forest terminé")

Validation croisée Random Forest...
ROC AUC RF: 0.7323 (+/- 0.0023)
Entraînement final...
 Random Forest terminé


In [29]:
# LightGBM
import lightgbm as lgb

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 15,
    'max_depth': 5,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'verbose': -1 # Pas de logs
}

lgb_train = lgb.Dataset(X, label=y)
model_lgb = lgb.train(lgb_params, lgb_train, num_boost_round=100)

# X_test vs X.columns car X.columns ne contient pas SK_ID_CURR
predictions_lgb = model_lgb.predict(X_test)  

print(" LightGBM terminé")

 LightGBM terminé


In [ ]:
# CatBoost
from catboost import CatBoostClassifier
from sklearn.model_selection import KFold
import numpy as np

# Créer le modèle
cat = CatBoostClassifier(
    iterations=100, 
    depth=5, 
    learning_rate=0.1, 
    random_seed=42,
    verbose=0  # Supprime les logs
)

# Validation croisée MANUELLE (3 folds)
kf = KFold(n_splits=3, shuffle=True, random_state=42)
cv_scores_cat = []

print("Validation croisée CatBoost...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"  Fold {fold}/3...", end='')
    
    # Séparer train/validation
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # Entraîner
    cat.fit(X_train_fold, y_train_fold, verbose=0)
    
    # Prédire sur validation
    y_pred_proba = cat.predict_proba(X_val_fold)[:, 1]
    
    # Calculer ROC AUC
    from sklearn.metrics import roc_auc_score
    score = roc_auc_score(y_val_fold, y_pred_proba)
    cv_scores_cat.append(score)
    
    print(f" ROC AUC = {score:.4f}")

# Résultat final
cv_scores_cat = np.array(cv_scores_cat)
print(f"\nROC AUC CatBoost: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std():.4f})")

# Entraînement final sur tout le dataset
print("Entraînement final sur toutes les données...")
cat.fit(X, y, verbose=0)
predictions_cat = cat.predict_proba(X_test)[:, 1]

print(" CatBoost terminé")

Validation croisée CatBoost...
  Fold 1/3... ROC AUC = 0.7645
  Fold 2/3... ROC AUC = 0.7652
  Fold 3/3... ROC AUC = 0.7645

ROC AUC CatBoost: 0.7647 (+/- 0.0003)
Entraînement final sur toutes les données...
✅ CatBoost terminé


In [ ]:
# XGBoost
import xgboost as xgb
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbosity': 0  # Pas de logs
}
xgb_train = xgb.DMatrix(X, label=y)
model_xgb = xgb.train(xgb_params, xgb_train, num_boost_round=100)

predictions_xgb = model_xgb.predict(xgb.DMatrix(X_test))  # X_test au lieu de test[X.columns]

print(" XGBoost terminé")

✅ XGBoost terminé


In [37]:
# sauvegardes
# Récupérer les SK_ID_CURR depuis test
test_ids = test['SK_ID_CURR']

# Logistic Regression
pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': predictions_lr
}).to_csv('submission_lr.csv', index=False)
print(" submission_lr.csv")

# Random Forest
pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': predictions_rf
}).to_csv('submission_rf.csv', index=False)
print(" submission_rf.csv")

# LightGBM
if predictions_lgb is not None:
    pd.DataFrame({
        'SK_ID_CURR': test_ids,
        'TARGET': predictions_lgb
    }).to_csv('submission_lgb.csv', index=False)
    print(" submission_lgb.csv")

# CatBoost
if predictions_cat is not None:
    pd.DataFrame({
        'SK_ID_CURR': test_ids,
        'TARGET': predictions_cat
    }).to_csv('submission_cat.csv', index=False)
    print(" submission_cat.csv")

# XGBoost
if predictions_xgb is not None:
    pd.DataFrame({
        'SK_ID_CURR': test_ids,
        'TARGET': predictions_xgb
    }).to_csv('submission_xgb.csv', index=False)
    print(" submission_xgb.csv")

print("\n Tous les modèles terminés !")




 submission_lr.csv
 submission_rf.csv
 submission_lgb.csv
 submission_cat.csv
 submission_xgb.csv

 Tous les modèles terminés !


In [41]:
print(" Submission sauvegardée")
print(f"\n ROC AUC Logistic Regression: {cv_scores_lr.mean():.4f}")
print(f"\n ROC AUC Random Forest: {cv_scores_rf.mean():.4f}")
print(f"\n ROC AUC LightGBM: ")
print(f"\n ROC AUC CatBoost: {cv_scores_cat.mean():.4f}")
print(f"\n ROC AUC XGBoost: ")


print(f" Features utilisées: {X.shape[1]}")

 Submission sauvegardée

 ROC AUC Logistic Regression: 0.6731

 ROC AUC Random Forest: 0.7323

 ROC AUC LightGBM: 

 ROC AUC CatBoost: 0.7647

 ROC AUC XGBoost: 
 Features utilisées: 58
